### Comparação de classificadores na CTG

Este notebook escolhe os hiperparâmetros de quatro classificadores da statstech (regressão logística, LDA, QDA e KNN) por validaçaõ cruzada e compara-os. 


#### 0. Configuração inicial 

In [12]:
import csv
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from statstech.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis
)
from statstech.linear_model import LogisticRegression
from statstech.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score
)
from statstech.neighbors import KNeighborsClassifier
from statstech.pipeline import Pipeline
from statstech.preprocessing import StandardScaler

from analysis.config import FIGURES, PROCESSED, SEED, STYLE, TABLES
from analysis.evaluation import evaluate_classifier
from analysis.io import load_dataset
from analysis.validation import StratifiedGroupKFold

plt.style.use(STYLE)

warnings.filterwarnings("ignore", message=".*indefinid.*")

CLASS_NAMES = ["normal", "suspeito", "patológico"]
PATHOLOGICAL = 2

#### 1. Dados e grupos do treino

ctg.npz traz treino e teste já separados por registro. Para a validação cruzada também precisamos saber de qual registro veio cada dado do treino.

In [13]:
data = load_dataset("ctg")
X_train, y_train = data["X_train"], data["y_train"]
features_names = data["feature_names"].tolist()

groups_train = np.load(PROCESSED / "ctg_groups_train.npy")
assert len(groups_train) == len(y_train)

print(f"treino: {X_train.shape[0]} exames, {X_train.shape[1]} medidas, "
      f"{len(np.unique(groups_train))} registros")
for k, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train == k)
    print(f"  {name:11s} {n:5d}  ({n / len(y_train):.1%})")

treino: 1755 exames, 20 medidas, 282 registros
  normal       1363  (77.7%)
  suspeito      245  (14.0%)
  patológico    147  (8.4%)


#### 2. Validação cruzada por registro

Segmentos do mesmo registro são muito parecidos. Se um cai no treino de um fold e outro na validação, o modelo "reconhece" o vizinho e a nota sobe sem o modelo ter aprendido nada que generalize. 


In [14]:
cv = StratifiedGroupKFold(groups_train, n_splits=10, random_state=SEED)

##### 2.1 KNN com folds por linha e por registro 

Busca por k com duas validações diferentes, mas com fold por linha segmentos "irmãos" se misturam entre treino e validação.

In [15]:
def make_pipeline(model):
    return Pipeline([("scaler", StandardScaler()), ("model", model)])

ks = [1, 5, 15, 31, 61]
cv_by_row = StratifiedKFold(10, shuffle=True, random_state=SEED)
for label, splitter in [("folds por linha", cv_by_row), ("folds por registro", cv)]:
    search = GridSearchCV(make_pipeline(KNeighborsClassifier()), {"model__n_neighbors": ks},
                          cv=splitter, scoring="f1_macro").fit(X_train, y_train)
    print(f"{label:19s} melhor k = {search.best_params_['model__n_neighbors']:2d}   "
          f"F1 por k: {[round(r['mean_score'], 3) for r in search.cv_results_]}")

folds por linha     melhor k =  1   F1 por k: [0.834, 0.804, 0.786, 0.762, 0.722]
folds por registro  melhor k = 15   F1 por k: [0.694, 0.722, 0.723, 0.704, 0.663]
